# União de tabelas similares
Monta uma única tabela com todas as tabelas com o mesmo padrão de nome.  
Inclui todas as colunas de todas as tabelas e as que não tiverem alguma coluna que outra tenha, fica com o valor NULL.

In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl.metadata
  Obtaining dependency information for et-xmlfile from https://files.pythonhosted.org/packages/c1/8b/5fe2cc11fee489817272089c4203e679c63b570a5aaeb18d852ae3cbba6a/et_xmlfile-2.0.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/250.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 153.6/250.9 kB 4.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


  Obtaining dependency information for xlsxwriter from https://files.pythonhosted.org/packages/3a/0c/3662f4a66880196a590b202f0db82d919dd2f89e99a27fadef91c4a33d41/xlsxwriter-3.2.9-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/175.3 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 153.6/175.3 kB 4.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


  Obtaining dependency information for unidecode from https://files.pythonhosted.org/packages/8f/b7/559f59d57d18b44c6d1250d2eeaa676e028b9c527431f5d0736478a73ba1/Unidecode-1.4.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/235.8 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 174.1/235.8 kB 5.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.5 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.types import DoubleType, LongType, TimestampType, StringType

In [0]:
# Definir o schema e o regex do nome das tabelas
nome_tabelas = "'tb_fecham_trab%'"
nome_schema = "databox.juridico_comum"
nome_tabela_unida = "geral_fechamento_trab"

### 1 - Definição das funções

In [0]:
# Carrega as tabelas em DF e retorna uma lista de DFs
def load_dataframes(schema, table_names):
    """
    Carrega as tabelas em DF e retorna uma lista de DFs
    """
    dataframes = []
    for table_name in table_names:
        df = spark.table(f"{schema}.{table_name}")
        dataframes.append(df)
    return dataframes

In [0]:
# Deduplica as colunas com mesmo nome
def deduplicate_columns(df: DataFrame) -> DataFrame:
    columns = df.columns
    renamed_columns = [f"{col}@{i}" for i, col in enumerate(columns)]
    
    # Rename columns to make them unique
    df_renamed = df.toDF(*renamed_columns)
    
    # Identify and keep the first occurrence of each column
    unique_columns = []
    seen = set()
    for i, col in enumerate(columns):
        if col not in seen:
            unique_columns.append(renamed_columns[i])
            seen.add(col)
    
    # Select only the unique columns
    df_unique = df_renamed.select(*unique_columns)
    
    # Optionally, rename columns back to original names, if needed
    original_names = [col.split('@')[0] for col in unique_columns]
    df_final = df_unique.toDF(*original_names)
    
    return df_final

### 2 - Execução

In [0]:

# Lista todas as tabelas no catálogo com o nome buscado
tables = spark.sql(f"SHOW TABLES IN {nome_schema}").collect()
tables = spark.createDataFrame(tables)

tables = tables.where(f"tableName LIKE {nome_tabelas}")

table_names = [row.tableName for row in tables.select('tableName').collect()]

table_names.sort(reverse = True)

print(table_names)

['tb_fecham_trab_202601', 'tb_fecham_trab_202512', 'tb_fecham_trab_202511', 'tb_fecham_trab_202510', 'tb_fecham_trab_202509', 'tb_fecham_trab_202508', 'tb_fecham_trab_202507', 'tb_fecham_trab_202506', 'tb_fecham_trab_202505', 'tb_fecham_trab_202504', 'tb_fecham_trab_202503', 'tb_fecham_trab_202502', 'tb_fecham_trab_202501', 'tb_fecham_trab_202412', 'tb_fecham_trab_202411', 'tb_fecham_trab_202410', 'tb_fecham_trab_202409', 'tb_fecham_trab_202408', 'tb_fecham_trab_202407', 'tb_fecham_trab_202406', 'tb_fecham_trab_202405', 'tb_fecham_trab_202404', 'tb_fecham_trab_202403', 'tb_fecham_trab_202402', 'tb_fecham_trab_202401', 'tb_fecham_trab_202312', 'tb_fecham_trab_202311', 'tb_fecham_trab_202310', 'tb_fecham_trab_202309', 'tb_fecham_trab_202308', 'tb_fecham_trab_202307', 'tb_fecham_trab_202306', 'tb_fecham_trab_202305', 'tb_fecham_trab_202304', 'tb_fecham_trab_202303', 'tb_fecham_trab_202302', 'tb_fecham_trab_202301', 'tb_fecham_trab_202212', 'tb_fecham_trab_202211', 'tb_fecham_trab_202210',

#### 2.1 Nomes das colunas

In [0]:
lista_dfs = load_dataframes(nome_schema, table_names)

# Ajusta os nomes de colunas para remover exesso de espaços e "_"
lista_dfs_clean = []
for df in lista_dfs:
    df_clean = remove_acentos(df)
    df_clean = adjust_column_names(df_clean)
    lista_dfs_clean.append(df_clean)

In [0]:
################## PRINTAR ANDAMENTO #############
lista_dfs = load_dataframes(nome_schema, table_names)

# Ajusta os nomes de colunas para remover exesso de espaços e "_"
lista_dfs_clean = []
for i, df in enumerate(lista_dfs):
    df_clean = remove_acentos(df)
    df_clean = adjust_column_names(df_clean)
    lista_dfs_clean.append(df_clean)
    
    print(table_names[i] + " concluida!")

tb_fecham_trab_202601 concluida!
tb_fecham_trab_202512 concluida!
tb_fecham_trab_202511 concluida!
tb_fecham_trab_202510 concluida!
tb_fecham_trab_202509 concluida!
tb_fecham_trab_202508 concluida!
tb_fecham_trab_202507 concluida!
tb_fecham_trab_202506 concluida!
tb_fecham_trab_202505 concluida!
tb_fecham_trab_202504 concluida!
tb_fecham_trab_202503 concluida!
tb_fecham_trab_202502 concluida!
tb_fecham_trab_202501 concluida!
tb_fecham_trab_202412 concluida!
tb_fecham_trab_202411 concluida!
tb_fecham_trab_202410 concluida!
tb_fecham_trab_202409 concluida!
tb_fecham_trab_202408 concluida!
tb_fecham_trab_202407 concluida!
tb_fecham_trab_202406 concluida!
tb_fecham_trab_202405 concluida!
tb_fecham_trab_202404 concluida!
tb_fecham_trab_202403 concluida!
tb_fecham_trab_202402 concluida!
tb_fecham_trab_202401 concluida!
tb_fecham_trab_202312 concluida!
tb_fecham_trab_202311 concluida!
tb_fecham_trab_202310 concluida!
tb_fecham_trab_202309 concluida!
tb_fecham_trab_202308 concluida!
tb_fecham_

#### 2.2 Deduplica

In [0]:
# Deduplica colunas com mesmo nome
lista_dfs_clean_dd = []
for i, df in enumerate(lista_dfs_clean):
    df_dd = deduplicate_columns(df)
    lista_dfs_clean_dd.append(df_dd)

    print(table_names[i] + " concluida!")

tb_fecham_trab_202601 concluida!
tb_fecham_trab_202512 concluida!
tb_fecham_trab_202511 concluida!
tb_fecham_trab_202510 concluida!
tb_fecham_trab_202509 concluida!
tb_fecham_trab_202508 concluida!
tb_fecham_trab_202507 concluida!
tb_fecham_trab_202506 concluida!
tb_fecham_trab_202505 concluida!
tb_fecham_trab_202504 concluida!
tb_fecham_trab_202503 concluida!
tb_fecham_trab_202502 concluida!
tb_fecham_trab_202501 concluida!
tb_fecham_trab_202412 concluida!
tb_fecham_trab_202411 concluida!
tb_fecham_trab_202410 concluida!
tb_fecham_trab_202409 concluida!
tb_fecham_trab_202408 concluida!
tb_fecham_trab_202407 concluida!
tb_fecham_trab_202406 concluida!
tb_fecham_trab_202405 concluida!
tb_fecham_trab_202404 concluida!
tb_fecham_trab_202403 concluida!
tb_fecham_trab_202402 concluida!
tb_fecham_trab_202401 concluida!
tb_fecham_trab_202312 concluida!
tb_fecham_trab_202311 concluida!
tb_fecham_trab_202310 concluida!
tb_fecham_trab_202309 concluida!
tb_fecham_trab_202308 concluida!
tb_fecham_

#### 2.3 Cast string

In [0]:
# Cast string todas as colunas
lista_dfs_clean_dd_string = []
for i, df in enumerate(lista_dfs_clean_dd):
    for col in df.columns:
        df = df.withColumn(col, df[col].cast(StringType()))
    
    lista_dfs_clean_dd_string.append(df)
    print(table_names[i] + " concluida!")
# lista_dfs_clean_dd_string[0].printSchema()

tb_fecham_trab_202601 concluida!
tb_fecham_trab_202512 concluida!
tb_fecham_trab_202511 concluida!
tb_fecham_trab_202510 concluida!
tb_fecham_trab_202509 concluida!
tb_fecham_trab_202508 concluida!
tb_fecham_trab_202507 concluida!
tb_fecham_trab_202506 concluida!
tb_fecham_trab_202505 concluida!
tb_fecham_trab_202504 concluida!
tb_fecham_trab_202503 concluida!
tb_fecham_trab_202502 concluida!
tb_fecham_trab_202501 concluida!
tb_fecham_trab_202412 concluida!
tb_fecham_trab_202411 concluida!
tb_fecham_trab_202410 concluida!
tb_fecham_trab_202409 concluida!
tb_fecham_trab_202408 concluida!
tb_fecham_trab_202407 concluida!
tb_fecham_trab_202406 concluida!
tb_fecham_trab_202405 concluida!
tb_fecham_trab_202404 concluida!
tb_fecham_trab_202403 concluida!
tb_fecham_trab_202402 concluida!
tb_fecham_trab_202401 concluida!
tb_fecham_trab_202312 concluida!
tb_fecham_trab_202311 concluida!
tb_fecham_trab_202310 concluida!
tb_fecham_trab_202309 concluida!
tb_fecham_trab_202308 concluida!
tb_fecham_

In [0]:
# for i, df in lista_dfs_clean_dd_string:
#     print(table_names[i] + " concluida!")

# print(lista_dfs_clean_dd_string)

#### 2.4 All schema

In [0]:
def get_all_schema(dfs: list[DataFrame]) -> StructType:
    """
    Pega a lista de dfs e retorna o schema em comum. Ou seja,
    todas as colunas presentes em todas as tabelas.
    """
    # Get the schemas of all DataFrames
    schemas = [df.schema.fields for df in dfs]
    
    # Flatten the list of fields and use a set to find unique fields
    all_fields = {field for schema in schemas for field in schema}
    
    # Create a StructType with all the unique fields
    all_schema = StructType(list(all_fields))
    
    return all_schema

In [0]:
def create_empty_df_with_all_schema(dfs: list[DataFrame]) -> DataFrame:
    """"
    Usa as colunas em comum para criar um DF vazio.
    """
    # Get the all-inclusive schema
    all_schema = get_all_schema(dfs)
    
    # Create an empty DataFrame with the all-inclusive schema
    empty_df = spark.createDataFrame([], all_schema)
    
    return empty_df

In [0]:
unified_df = create_empty_df_with_all_schema(lista_dfs_clean_dd_string)

In [0]:
display(unified_df)

CA,PAGAMENTOS,TERCEIRO,CLUSTER_VALOR92,INDICACAO_PROCESSO_ESTRATEGICO,CLUSTER_AGING95,SUB_OBJETO_ASSUNTO_CARGO_M,DH,SAFRA_DE_RECLAMACAO96,SOCIO:_PROVISAO_TOTAL_M,EMPRESA:_CORRECAO_M_1,PARCELAMENTO_CONDENACAO,DEPARA_MAPA_MOV,FASE_M,SOCIO:_CORRECAO_M,ESTRATEGIA,PROVISAO_MOV_TOTAL_M,PARCELADO,SOCIO:_PROVISAO_M_1,SAFRA_DE_RECLAMACAO101,MES_FECH,DEMITIDO_POR_REESTRUTURACAO_RH,VALOR_INSS_DECAIDO,GARANTIA,%_EMPRESA_M_1,TIPO_DE_CALCULO_M_1,EMPRESA:_PROVISAO_M_1,STATUS_M,FASE_M_1,CLUSTER_AGING_2,CARGO_TRATADO103,PAGAMENTO_HIST,STATUS_M_1,ENCERRADOS,CLUSTER_AGING_TEMPO_DE_EMPRESA94,PARCELAMENTO_ACORDO,TIPO,REATIVADOS,SUM_OF_VALOR,%_SOCIO_M_1,IMPACTO,ORGAO_OFENSOR_FLUXO_M,VLR_CAUSA,VALIDADOR_PROVISAO_PELO_ADVOGADO,CLUSTER_AGING_TEMPO_DE_EMPRESA89,%_EMPRESA_M,CLASSIFICACAO_MOV_M,NOVOS,MOTIVO_PAGAMENTO,ELEGIVEL_M,CLUSTER_AGING100,LINHA,EMPRESA:_CORRECAO_M,<>2020_NOVO_RNO_ATUALIZACAO,NOVO_X_LEGADO,MEDIA_DE_PAGAMENTO,CENTRO_DE_CUSTO_M,CARGO_TRATADO93,DG,TOTAL_PAGAMENTOS,SAFRA_DE_RECLAMACAO_1,UF,NATUREZA_OPERACIONAL_M,CARTEIRA,CLUSTER_AGING_TEMPO_DE_EMPRESA99,CONDENACAO,CLUSTER_AGING_TEMPO_DE_EMPRESA_2,PROVISAO_TOTAL_PASSIVO_M,CLUSTER_VALOR_2,AREA_FUNCIONAL,CENTRO_DE_CUSTO_M_1,SOCIO:_PROVISAO_MOV_TOTAL_M,EMPRESA:_PROVISAO_TOTAL_M_1,EMPRESA:_CORRECAO_M_1_1,CARGO_TRATADO_1,OUTRAS_ADICOES,ANO_CADASTRO,CLUSTER_VALOR102,C_S_PGTO,EMPRESA:_CORRECAO_M_1_0001,PROVISAO_M_1,CORRECAO_M,SUB_TIPO,PROCESSO_COM_DOCUMENTO,EMPRESA:_PROVISAO_MOV_TOTAL_M,EMPRESA:_CORRECAO_MOV_M,ESTRATEGIA_M_1,NOVO_X_LEGADO87,SAFRA_DE_RECLAMACAO_2,TIPO_PAGAMENTO,PROVISAO_TOTAL_M_1,AREA_DO_DIREITO,VALIDADOR_PROVISAO_PELA_MODELAGEM,DF,OBSERVACOES,NATUREZA_OPERACIONAL_M_1,NOVO_X_LEGADO76,VALOR_DO_CALCULO,SAFRA_DE_RECLAMACAO,SOCIO:_PROVISAO_MOV_M,ESTOQUE,CORRECAO_M_1,PASTA,EMPRESA_M_1,DOC,TIPO_DE_CALCULO_M,CLASSIFICACAO_POR_POSICAO,SUB_OBJETO_ASSUNTO_CARGO_M_1,CLUSTER_AGING_TEMPO_DE_EMPRESA_1,COMPOE_PROVISAO,SOCIO:_CORRECAO_MOV_M,OUTRAS_REVERSOES,SUB_AREA_DO_DIREITO,SOCIO:_PROVISAO_TOTAL_M_1,ID_PROCESSO,SOCIO:_CORRECAO_M_1,SOCIO:_CLASSIFICACAO_MOV_M,NO_PROCESSO,REABERTURA,DEMITIDO_POR_REESTRUTURACAO,SAFRA_DE_RECLAMACAO91,EMPRESA:_CLASSIFICACAO_MOV_M,IMPOSTO,SOCIO:_CORRECAO_M_1_1,PROPRIO_X_TERCEIRO,EMPRESA:_PROVISAO_TOTAL_M,PGTO,CORRECAO_MOV_M,ACORDOS,VALOR_PAGAMENTO_HIST,CLUSTER_AGING_1,SOCIO:_PROV_TOTAL_PASSIVO_M,FILIAL_M,CLUSTER_AGING,MOTIVO_MOVIMENTACAO,%_RISCO,ESCRITORIO,DISTRIBUICAO,CLUSTER_VALOR_1,ORGAO_OFENSOR_FLUXO_M_1,DT_ULT_PGTO,CLUSTER_AGING90,CARGO_TRATADO98,CLUSTER_VALOR,%_SOCIO_M,PROVISAO_TOTAL_M,TIPO_PGTO,GRUPO_M,DP_FASE,EMPRESA_M,PROVISAO_MOV_M,GRUPO_FECHAMENTO,CLUSTER_VALOR97,GRUPO_M_1,ELEGIVEL_M_1,CARGO_TRATADO,VALOR_PAGAMENTO,EMPRESA:_PROV_TOTAL_PASSIVO_M,EMPRESA:_PROVISAO_MOV_M,CARGO_TRATADO_2,OBJETO_ASSUNTO_CARGO_M,DATACADASTRO,C91,PENHORA,DI,CLUSTER_AGING_TEMPO_DE_EMPRESA,OBJETO_ASSUNTO_CARGO_M_1,XX,COMPOE_RNO,SOCIO:_CORRECAO_M_1_0001,BZ,PROV,OUTROS_PAGAMENTOS


### 3 - Union

In [0]:
for df in lista_dfs_clean_dd_string:
    unified_df = unified_df.unionByName(df, allowMissingColumns=True)

# unified_df.display()

### 4 - Recast

In [0]:
# Cria um DF com seus tipos originais
df_ref = create_empty_df_with_all_schema(lista_dfs_clean_dd)

# Usa esse schema como referencia para converter novamente os tipos do DF unido
schema_ref = df_ref.schema

for field in schema_ref.fields:
    unified_df = unified_df.withColumn(field.name, unified_df[field.name].cast(field.dataType))

### 5 - Salva a tabela final


In [0]:
spark.sql(f"TRUNCATE TABLE {nome_schema}.{nome_tabela_unida}")

DataFrame[]

In [0]:
unified_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{nome_schema}.{nome_tabela_unida}")

In [0]:
unified_df.count()

1968404

In [0]:
dbutils.notebook.exit("Success")

In [0]:
%sql
SELECT 
    date_format(MES_FECH, 'yyyy-MM') as AnoMes,
    count(*) as Qtd_Linhas
FROM databox.juridico_comum.geral_fechamento_trab
GROUP BY 1
ORDER BY 1 DESC